## Experiment 8 – K-Means and Hierarchical Clustering

This experiment uses `/content/iris.csv`, as in the PDF.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.cluster.hierarchy import linkage, dendrogram

csv_path = Path("/content/iris.csv")
df = pd.read_csv(csv_path)

rename_dict = {
    "sepal_length": "SepalLengthCm",
    "sepal_width": "SepalWidthCm",
    "petal_length": "PetalLengthCm",
    "petal_width": "PetalWidthCm",
    "species": "Species"
}

df = df.rename(columns=rename_dict)

print("Shape:", df.shape)
print(df.head())
print("Missing values:\n", df.isna().sum())
print("Species counts:\n", df["Species"].value_counts())

In [ ]:
features = ["SepalLengthCm", "SepalWidthCm", "PetalLengthCm", "PetalWidthCm"]
required = features + ["Species"]

if not set(required).issubset(df.columns):
    raise ValueError("Check CSV column names")

if df[required].isna().any().any():
    raise ValueError("Handle missing values first.")

X = df[features].to_numpy(dtype=float)
true_species = df["Species"].to_numpy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Feature shape:", X.shape)

In [ ]:
kmeans = KMeans(n_clusters=3, init="k-means++", n_init=10, random_state=42)
kmeans_labels = kmeans.fit_predict(X_scaled)

print(pd.Series(kmeans_labels).value_counts().sort_index())
print("Inertia:", kmeans.inertia_)

centers = scaler.inverse_transform(kmeans.cluster_centers_)
print(pd.DataFrame(centers, columns=features).round(2))

In [ ]:
plt.figure(figsize=(7, 5))

points = plt.scatter(X[:, 2], X[:, 3], c=kmeans_labels, cmap="viridis", edgecolors="black")
plt.scatter(centers[:, 2], centers[:, 3], c="red", marker="X", s=200, label="Centers")
plt.xlabel("Petal length (cm)")
plt.ylabel("Petal width (cm)")
plt.title("K-means clustering")
plt.colorbar(points, ticks=[0, 1, 2], label="Cluster")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

linkage_matrix = linkage(X_scaled, method="ward")
dendrogram(linkage_matrix, labels=true_species, leaf_rotation=90, leaf_font_size=8)
plt.title("Hierarchical Clustering Dendrogram (Ward Linkage)")
plt.xlabel("Sample index or class")
plt.ylabel("Distance")
plt.tight_layout()
plt.show()